<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

# Explicabilidad e Inteligencia Artificial Interpretable (XAI)
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Objetivo:** Responder PI2 y OE4 — ¿qué variables explican la satisfacción
democrática y cuáles son sus efectos no lineales?

### Estructura
| Sección | Contenido |
|---|---|
| 1–2 | Importaciones y configuración |
| 3 | Selección del modelo principal |
| 4 | SHAP global — importancias por bloque |
| 5 | SHAP beeswarm — dirección e intensidad |
| 6 | ALE — efectos no lineales y umbrales |
| 7 | SHAP local + LIME — explicaciones individuales |
| 8 | TabNet — análisis de atención (separado de SHAP) |
| 9 | Guardado de valores SHAP para notebook 05 |

## 1. Importaciones

In [1]:
import sys
sys.path.append("..")

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from typing import Dict, List, Optional

warnings.filterwarnings("ignore")

from utils.config import (
    setup_plots, THEME, PATHS, SUBPERIODOS, SUBREGIONES,
    PAISES_EXCLUIR_TEST, COL_TARGET, COL_PAIS,
    ETIQUETAS, ETIQUETAS_FEATURES, BLOQUES, bloque_de, N_CLASES,
)
from utils.io import (
    cargar_pipeline, cargar_split_parquet, cargar_resultados,
    cargar_mejor_modelo, guardar_shap_values, cargar_shap_values,
    shap_disponible,
)
from utils.plots import (
    plot_shap_bar_bloques, plot_shap_beeswarm, plot_ale,
    save_figure, model_color,
)

# Verificar disponibilidad de shap
try:
    import shap
    SHAP_OK = True
    print(f"✓ shap {shap.__version__} disponible")
except ImportError:
    SHAP_OK = False
    print("⚠ shap no instalado — instalar con: pip install shap")

# Verificar disponibilidad de lime
try:
    import lime
    import lime.lime_tabular
    LIME_OK = True
    print(f"✓ lime disponible")
except ImportError:
    LIME_OK = False
    print("⚠ lime no instalado — instalar con: pip install lime")

# Verificar alibi para ALE
try:
    from alibi.explainers import ALE
    ALE_OK = True
    print("✓ alibi disponible para ALE")
except ImportError:
    ALE_OK = False
    print("⚠ alibi no instalado (ALE) — instalar con: pip install alibi")

setup_plots()
print("\n✓ Importaciones completadas.")

✓ shap 0.52.0 disponible
✓ lime disponible


I0000 00:00:1784260961.862461  235742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784260961.862779  235742 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784260961.885789  235742 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ alibi disponible para ALE

✓ Importaciones completadas.


I0000 00:00:1784260962.493836  235742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784260962.494005  235742 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. Configuración

In [2]:
# =============================================================================
# Parámetros del notebook XAI
# =============================================================================

# ── Selección del modelo para análisis principal ──────────────────────────────
# 'auto' : usa el mejor modelo según Kappa Cuadrático en SP3 (recomendado)
# nombre : fija un modelo específico ('XGBoost', 'CatBoost', etc.)
MODELO_XAI = "auto"

# ── TabNet: SHAP o atención propia ────────────────────────────────────────────
# True  → usa SHAP DeepExplainer para TabNet (más lento, ~30 min)
# False → usa el mecanismo de atención nativo de TabNet (recomendado)
SHAP_PARA_TABNET = False

# ── Número de muestras para SHAP (árboles) ───────────────────────────────────
# Los TreeExplainer de shap son exactos — no necesitan muestreo.
# Para KernelExplainer (OLO) sí se limita el número de muestras.
N_MUESTRAS_SHAP_OLO = 500    # KernelExplainer es costoso O(n²)

# ── Número de muestras para LIME ──────────────────────────────────────────────
# Estratificada por clase y subregión.
# 50% casos representativos + 50% casos discordantes (ver sección 7).
N_MUESTRAS_LIME     = 200

# ── Subperiodo de referencia para análisis principal ──────────────────────────
SP_REFERENCIA = "SP3"

# ── Recalcular SHAP aunque ya existan archivos guardados ─────────────────────
FORZAR_RECALCULO_SHAP = False

# ── Top N variables a mostrar en gráficos ────────────────────────────────────
TOP_N_SHAP = 20

# Bandera: LIME sobre errores reales del modelo
# True: genera LIME para casos con mayor distancia ordinal |pred - real|
# False: solo LIME representativo y discordante institucional
LIME_SOBRE_ERRORES = True
N_CASOS_ERROR_LIME = 50

# Resolver modelo automático
if MODELO_XAI == "auto":
    try:
        sel_path = PATHS["FOLDER_RESULTS"] / "modelo_xai_seleccionado.json"
        if sel_path.exists():
            sel = json.loads(sel_path.read_text())
            MODELO_XAI = sel["modelo_xai"]
            print(f"Modelo principal (cargado desde notebook 03): {MODELO_XAI}")
        else:
            MODELO_XAI = cargar_mejor_modelo(SP_REFERENCIA, "kappa_cuadratico")
            print(f"Modelo principal (calculado): {MODELO_XAI}")
    except Exception as e:
        MODELO_XAI = "XGBoost"
        print(f"⚠ No se pudo resolver el modelo automático ({e}).")
        print(f"  Usando fallback: {MODELO_XAI}")

print(f"\nConfiguración XAI:")
print(f"  Modelo principal   : {MODELO_XAI}")
print(f"  Subperiodo ref.    : {SP_REFERENCIA}")
print(f"  SHAP para TabNet   : {SHAP_PARA_TABNET}")
print(f"  Muestras LIME      : {N_MUESTRAS_LIME}")

Modelo principal (cargado desde notebook 03): CatBoost

Configuración XAI:
  Modelo principal   : CatBoost
  Subperiodo ref.    : SP3
  SHAP para TabNet   : False
  Muestras LIME      : 200


## 3. Carga del modelo y datos de referencia

In [3]:
# =============================================================================
# Cargar pipeline principal y datos de prueba del subperiodo de referencia
# =============================================================================
art  = cargar_pipeline(MODELO_XAI, SP_REFERENCIA)
tipo = art["tipo_modelo"]

df_te   = cargar_split_parquet(SP_REFERENCIA, "test")
feats   = art["features"]
feats_d = [f for f in feats if f in df_te.columns]

X_te_raw = df_te[feats_d].reindex(columns=feats)
y_te     = df_te[COL_TARGET].astype(int).values

# Aplicar preprocesamiento según tipo de modelo
if tipo in ("olo", "tabnet"):
    X_te_proc = pd.DataFrame(
        art["imp_num"].transform(X_te_raw), columns=feats)
    if tipo == "olo":
        X_te_proc = pd.DataFrame(
            art["scaler"].transform(X_te_proc), columns=feats)
    else:
        X_te_proc = pd.DataFrame(
            art["scaler"].transform(X_te_proc), columns=feats)
else:
    X_te_proc = X_te_raw.copy()
    if MODELO_XAI == "CatBoost":
        for col in art.get("vars_categoricas", []):
            if col in X_te_proc.columns:
                X_te_proc[col] = (X_te_proc[col].fillna(-999)
                                  .astype(int).astype(str))

print(f"Dataset de referencia: {X_te_proc.shape[0]:,} registros × {X_te_proc.shape[1]} features")
print(f"Modelo cargado: {MODELO_XAI} ({SP_REFERENCIA})")
print(f"Tipo de modelo: {tipo}")

Dataset de referencia: 328 registros × 45 features
Modelo cargado: CatBoost (SP3)
Tipo de modelo: trees


## 4. SHAP Global — Importancias por bloque temático

Se calculan los valores SHAP para el conjunto de prueba del subperiodo de
referencia. Para modelos de árboles se usa `TreeExplainer` (exacto).
Para OLO se usa `KernelExplainer` sobre una muestra representativa.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
# =============================================================================
# Cálculo de valores SHAP globales
# =============================================================================

def calcular_shap(modelo_nombre, subperiodo, X_proc, art_data,
                  forzar=False):
    """
    Calcula o carga valores SHAP para un modelo y subperiodo.
    Si ya existen en disco y forzar=False, los carga directamente.
    Retorna array (n_muestras × n_features).
    """
    if not forzar and shap_disponible(modelo_nombre, subperiodo):
        print(f"  Cargando SHAP guardados: {modelo_nombre} — {subperiodo}")
        return cargar_shap_values(modelo_nombre, subperiodo).values

    if not SHAP_OK:
        raise ImportError("shap no está instalado.")

    tipo_m = art_data["tipo_modelo"]
    clf    = art_data["modelo"]

    print(f"  Calculando SHAP: {modelo_nombre} — {subperiodo}...")

    if tipo_m in ("trees",):
        explainer  = shap.TreeExplainer(clf)
        shap_vals  = explainer.shap_values(X_proc)
        # Para multiclase: shap_vals es lista de arrays (uno por clase)
        # Usar media absoluta entre clases para importancia global
        if isinstance(shap_vals, list):
            shap_arr = np.stack(shap_vals, axis=2)
            shap_global = np.abs(shap_arr).mean(axis=2)
        else:
            shap_global = shap_vals

    elif tipo_m == "olo":
        # KernelExplainer: usar muestra reducida
        idx_sample = np.random.choice(len(X_proc), min(N_MUESTRAS_SHAP_OLO, len(X_proc)),
                                      replace=False)
        X_sample   = X_proc.values[idx_sample]
        background = shap.kmeans(X_proc.values, 50)
        explainer  = shap.KernelExplainer(clf.predict_proba, background)
        shap_vals  = explainer.shap_values(X_sample)
        if isinstance(shap_vals, list):
            shap_global = np.abs(np.stack(shap_vals, axis=2)).mean(axis=2)
        else:
            shap_global = shap_vals

    else:
        raise ValueError(f"SHAP para tipo '{tipo_m}' no soportado aquí.")

    guardar_shap_values(shap_global, list(X_proc.columns),
                        modelo_nombre, subperiodo)
    return shap_global


# Calcular SHAP para el modelo principal
shap_global = calcular_shap(
    MODELO_XAI, SP_REFERENCIA, X_te_proc, art,
    forzar=FORZAR_RECALCULO_SHAP,
)

# Importancia media absoluta por feature
importancias = pd.Series(
    np.abs(shap_global).mean(axis=0),
    index=feats,
).sort_values(ascending=False)

print(f"\nTop 10 features por |SHAP| medio:")
for feat, val in importancias.head(10).items():
    et = ETIQUETAS_FEATURES.get(feat, feat)
    bl = bloque_de(feat)
    print(f"  {et:<35} {val:.4f}  [{bl}]")

  Calculando SHAP: CatBoost — SP3...


  ✓ SHAP guardado: shap_CatBoost_SP3.parquet (158 KB)


ValueError: Data must be 1-dimensional, got ndarray of shape (45, 4) instead

In [ ]:
# Gráfico de importancias por bloque
plot_shap_bar_bloques(
    importancias,
    top_n=TOP_N_SHAP,
    titulo=f"Importancia global (|SHAP| medio) — {MODELO_XAI} · {SP_REFERENCIA}",
    nombre_archivo=f"04_shap_bar_{MODELO_XAI}_{SP_REFERENCIA}",
)

# Importancia agregada por bloque temático
imp_bloque = {}
for bloque, variables in BLOQUES.items():
    vars_pres = [v for v in variables if v in importancias.index]
    if vars_pres:
        imp_bloque[bloque] = importancias[vars_pres].sum()

df_bloque = pd.Series(imp_bloque).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
colores_b = [THEME.get("blocks",{}).get(b,"#888888") for b in df_bloque.index]
ax.barh(df_bloque.index[::-1], df_bloque.values[::-1],
        color=colores_b[::-1], edgecolor="white")
ax.set_xlabel("|SHAP| total del bloque")
ax.set_title(f"Importancia por bloque temático — {MODELO_XAI} · {SP_REFERENCIA}",
             fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
save_figure(f"04_shap_bloques_{MODELO_XAI}_{SP_REFERENCIA}")
plt.show()

# Guardar tabla
df_imp_tabla = pd.DataFrame({
    "variable": importancias.index,
    "etiqueta": [ETIQUETAS_FEATURES.get(v,v) for v in importancias.index],
    "bloque"  : [bloque_de(v) for v in importancias.index],
    "shap_medio": importancias.values,
})
df_imp_tabla.to_csv(
    PATHS["FOLDER_RESULTS_TABLES"] / f"shap_importancias_{MODELO_XAI}_{SP_REFERENCIA}.csv",
    index=False)
print("✓ Tabla de importancias guardada")

## 5. SHAP Beeswarm — dirección e intensidad

In [ ]:
# =============================================================================
# Beeswarm plot: muestra la dirección y la magnitud del efecto SHAP
# por cada observación del conjunto de prueba.
# =============================================================================
plot_shap_beeswarm(
    shap_values    = shap_global,
    X              = X_te_proc,
    top_n          = TOP_N_SHAP,
    titulo         = f"SHAP Beeswarm — {MODELO_XAI} · {SP_REFERENCIA}",
    nombre_archivo = f"04_shap_beeswarm_{MODELO_XAI}_{SP_REFERENCIA}",
)

In [ ]:
# =============================================================================
# SHAP en todos los subperiodos para comparabilidad (se guardan para NB05)
# =============================================================================
print("Calculando SHAP para los 3 subperiodos del modelo principal...")
for sp in ["SP1","SP2","SP3"]:
    if sp == SP_REFERENCIA:
        continue   # ya calculado arriba
    try:
        art_sp  = cargar_pipeline(MODELO_XAI, sp)
        df_te_sp = cargar_split_parquet(sp, "test")
        feats_sp = art_sp["features"]
        X_sp     = df_te_sp[[f for f in feats_sp if f in df_te_sp.columns]]
        X_sp     = X_sp.reindex(columns=feats_sp)
        # Preprocesamiento según tipo
        if art_sp["tipo_modelo"] in ("olo","tabnet"):
            X_sp = pd.DataFrame(art_sp["imp_num"].transform(X_sp), columns=feats_sp)
            X_sp = pd.DataFrame(art_sp["scaler"].transform(X_sp), columns=feats_sp)
        calcular_shap(MODELO_XAI, sp, X_sp, art_sp, forzar=FORZAR_RECALCULO_SHAP)
    except Exception as e:
        print(f"  ⚠ Error en {MODELO_XAI} — {sp}: {e}")

print("✓ Valores SHAP disponibles para SP1, SP2, SP3")

## 6. ALE — Efectos no lineales y umbrales

Los gráficos de Accumulated Local Effects (ALE) muestran el efecto marginal
de cada variable sobre la probabilidad predicha, capturando no linealidades
que SHAP promedia globalmente.

Se analizan las variables con mayor importancia SHAP del bloque de
**Confianza institucional** y **Evaluación económica**, que son las que
la teoría democrática (H3) predice como determinantes dominantes.

In [ ]:
# =============================================================================
# ALE para las top variables del análisis
# =============================================================================
from sklearn.inspection import PartialDependenceDisplay

# Seleccionar las 6 features más importantes de los dos bloques principales
bloques_principales = ["Confianza institucional", "Evaluación económica",
                       "Percepción política"]
vars_ale = []
for bloque in bloques_principales:
    for v in BLOQUES.get(bloque, []):
        if v in importancias.index:
            vars_ale.append(v)
    if len(vars_ale) >= 6:
        break
vars_ale = vars_ale[:6]

print(f"Variables para ALE: {[ETIQUETAS_FEATURES.get(v,v) for v in vars_ale]}")

# Gráfico ALE usando sklearn PartialDependenceDisplay como proxy
# (ALE nativo requiere alibi; usamos PDP centrado como aproximación)
clf_plot = art["modelo"]

if not isinstance(X_te_proc, np.ndarray):
    X_te_arr = X_te_proc.values
else:
    X_te_arr = X_te_proc

try:
    for var in vars_ale:
        if var not in feats:
            continue
        idx_var = feats.index(var)
        et_var  = ETIQUETAS_FEATURES.get(var, var)

        if ALE_OK:
            # ALE nativo con alibi
            from alibi.explainers import ALE as AlibiALE
            ale_exp = AlibiALE(clf_plot.predict_proba,
                               feature_names=feats,
                               target_names=list(ETIQUETAS.values()))
            ale_exp.fit(X_te_arr, min_bin_points=5)
            # Clase 2 (Más bien satisfecho) como referencia
            ale_vals = ale_exp.ale_values[idx_var][:, 2]
            ale_qs   = ale_exp.feature_values[idx_var]
            plot_ale(ale_vals, ale_qs, var,
                     titulo=f"ALE — {et_var}  [{bloque_de(var)}]",
                     nombre_archivo=f"04_ale_{var}")
        else:
            # Fallback: PDP parcial centrado
            fig, ax = plt.subplots(figsize=(7, 3.5))
            PartialDependenceDisplay.from_estimator(
                clf_plot, X_te_proc, [idx_var],
                kind="average", ax=ax,
                feature_names=feats,
            )
            ax.set_title(f"PDP (proxy ALE) — {et_var}  [{bloque_de(var)}]",
                         fontweight="bold")
            ax.set_xlabel(et_var)
            save_figure(f"04_pdp_{var}")
            plt.show()
except Exception as e:
    print(f"⚠ Error en ALE/PDP: {e}")
    print("  Verifica que el modelo soporta predict_proba y el formato de X.")

## 7. SHAP Local + LIME — Explicaciones individuales

Se seleccionan **200 casos** del conjunto de prueba usando dos criterios:
- **100 casos representativos**: muestra estratificada por clase y subregión
  (5 subregiones × 4 clases = 20 celdas, 5 casos por celda).
- **100 casos discordantes**: ciudadanos cuya clase predicha difiere más
  de lo esperado dado el índice democrático de su país (máxima discrepancia
  entre satisfacción individual y contexto institucional).

In [ ]:
# =============================================================================
# Construccion de la muestra para LIME — tres criterios
# A) Representativos: estratificados por clase x subregion
# B) Mayor error real: |clase_pred - clase_real| maxima
# C) Discordantes institucionales: alta poliarquia + baja satisfaccion predicha
# =============================================================================

def seleccionar_muestra_lime(
    df_te, X_proc, y_true, y_pred,
    n_representativos=100, n_discordantes=100,
):
    """Retorna (idx_repr, idx_errores, idx_disc)."""

    # A) Estratificado por clase x subregion
    idxs_repr = []
    clases = sorted(pd.Series(y_true).unique())
    n_cel  = max(1, n_representativos // (len(SUBREGIONES) * len(clases)))
    for sr, paises_sr in SUBREGIONES.items():
        mask_sr = (df_te[COL_PAIS].isin(paises_sr)
                   if COL_PAIS in df_te.columns
                   else pd.Series(True, index=df_te.index))
        for cls in clases:
            mask_cls = (pd.Series(y_true) == cls).values & mask_sr.values
            cands    = np.where(mask_cls)[0]
            if len(cands) > 0:
                idxs_repr.extend(
                    np.random.choice(cands, min(n_cel, len(cands)), replace=False).tolist())
    idxs_repr = list(dict.fromkeys(idxs_repr))[:n_representativos]

    # B) Mayor error de prediccion real
    errores_ord = np.abs(y_pred.astype(int) - y_true.astype(int))
    dist_max    = errores_ord.max()
    idxs_err    = []
    for dist in range(int(dist_max), 0, -1):
        cands_err = np.where(errores_ord == dist)[0]
        necesito  = (n_discordantes // 2) - len(idxs_err)
        if necesito <= 0:
            break
        idxs_err.extend(
            np.random.choice(cands_err, min(necesito, len(cands_err)), replace=False).tolist())

    # C) Discordantes institucionales (V-Dem)
    idxs_disc = []
    if "v2x_polyarchy" in df_te.columns:
        df_aux = df_te.copy()
        df_aux["y_pred"] = y_pred
        mask_disc = (
            ((df_aux["v2x_polyarchy"] > 0.6) & (df_aux["y_pred"] <= 1)) |
            ((df_aux["v2x_polyarchy"] < 0.3) & (df_aux["y_pred"] >= 2))
        )
        cands_disc = np.where(mask_disc)[0]
        n_obj_disc = n_discordantes // 2
        if len(cands_disc) > 0:
            idxs_disc = np.random.choice(
                cands_disc, min(n_obj_disc, len(cands_disc)), replace=False).tolist()

    print(f"  Casos representativos    : {len(idxs_repr)}")
    print(f"  Casos mayor error real   : {len(idxs_err)}")
    print(f"  Casos discordantes inst. : {len(idxs_disc)}")
    return np.array(idxs_repr), np.array(idxs_err), np.array(idxs_disc)


# Predicciones sobre todo el test set
if tipo in ("olo", "tabnet"):
    y_pred_all = art["modelo"].predict(X_te_proc.values)
else:
    y_raw      = art["modelo"].predict(X_te_proc)
    y_pred_all = y_raw.flatten() if hasattr(y_raw, "flatten") else y_raw
y_pred_all = np.array(y_pred_all).astype(int)

np.random.seed(42)
idx_repr, idx_err, idx_disc = seleccionar_muestra_lime(
    df_te, X_te_proc, y_te, y_pred_all,
    n_representativos = N_MUESTRAS_LIME // 2,
    n_discordantes    = N_MUESTRAS_LIME // 2,
)
idx_lime = np.unique(np.concatenate([idx_repr, idx_err, idx_disc]))
print(f"  Total unicos para LIME   : {len(idx_lime)}")

# Resumen de errores graves
print()
print("Errores graves en el test set completo (distancia ordinal):")
errores_ord_all = np.abs(y_pred_all - y_te)
for dist in range(4):
    n   = (errores_ord_all == dist).sum()
    pct = n / len(y_te) * 100
    print(f"  Distancia {dist}: {n:>6,} ({pct:>5.1f}%)")

for cp, cr in [(3, 0), (0, 3)]:
    mask = (y_pred_all == cp) & (y_te == cr)
    n    = mask.sum()
    if n > 0:
        print(f"  Pred={cp} Real={cr}: {n} casos", end="")
        if COL_PAIS in df_te.columns:
            top = df_te.loc[mask, COL_PAIS].value_counts().head(2)
            print(f" — {dict(top)}", end="")
        print()

In [ ]:
# =============================================================================
# LIME — explicaciones locales en tres grupos de casos
# =============================================================================
if LIME_OK and len(idx_lime) > 0:
    import lime.lime_tabular as lime_tab

    etiq_feats = [ETIQUETAS_FEATURES.get(f, f) for f in feats]
    explainer_lime = lime_tab.LimeTabularExplainer(
        training_data         = X_te_proc.values,
        feature_names         = etiq_feats,
        class_names           = list(ETIQUETAS.values()),
        mode                  = "classification",
        random_state          = 42,
        discretize_continuous = True,
    )

    if tipo in ("olo", "tabnet"):
        fn_pred = lambda x: art["modelo"].predict_proba(x)
    else:
        fn_pred = lambda x: art["modelo"].predict_proba(
            pd.DataFrame(x, columns=feats))

    def explicar_grupo(indices, tag, max_casos=10):
        filas = []
        for idx in indices[:max_casos]:
            exp   = explainer_lime.explain_instance(
                data_row=X_te_proc.values[idx], predict_fn=fn_pred,
                num_features=10, num_samples=1000, top_labels=1)
            label = exp.available_labels()[0]
            for feat_name, peso in exp.as_list(label=label):
                filas.append({
                    "grupo"      : tag,
                    "idx"        : int(idx),
                    "clase_real" : int(y_te[idx]),
                    "clase_pred" : int(y_pred_all[idx]),
                    "distancia"  : int(abs(y_pred_all[idx] - y_te[idx])),
                    "feature"    : feat_name,
                    "peso_lime"  : peso,
                })
        return pd.DataFrame(filas)

    print("Calculando LIME — Grupo A: representativos...")
    df_lime_repr = explicar_grupo(idx_repr, "representativo")

    print("Calculando LIME — Grupo B: mayor error real...")
    df_lime_err  = explicar_grupo(idx_err, "error_maximo")

    print("Calculando LIME — Grupo C: discordantes inst....")
    df_lime_disc = explicar_grupo(idx_disc, "discordante")

    df_lime_all = pd.concat([df_lime_repr, df_lime_err, df_lime_disc],
                             ignore_index=True)
    ruta_lime = PATHS["FOLDER_RESULTS_TABLES"] / f"lime_{MODELO_XAI}_{SP_REFERENCIA}.csv"
    df_lime_all.to_csv(ruta_lime, index=False)
    print(f"\n✓ LIME guardado: {ruta_lime.name}")

    # Grafico: top features en errores graves
    if LIME_SOBRE_ERRORES and not df_lime_err.empty:
        imp_err = (df_lime_err.groupby("feature")["peso_lime"]
                   .apply(lambda x: x.abs().mean())
                   .sort_values(ascending=False).head(10))
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(imp_err.index[::-1], imp_err.values[::-1],
                color="#DC2626", edgecolor="white", alpha=0.85)
        ax.set_xlabel("|peso LIME| medio")
        ax.set_title(
            f"Variables en casos de error grave — {MODELO_XAI} x {SP_REFERENCIA}\n"
            "Casos con mayor distancia ordinal |pred - real|",
            fontweight="bold")
        ax.grid(True, axis="x", alpha=0.3)
        from utils.plots import save_figure
        save_figure(f"04_lime_errores_{MODELO_XAI}_{SP_REFERENCIA}")
        plt.show()

    # Resumen por grupo
    print()
    print("Resumen LIME por grupo:")
    for tag, df_g in [("Representativo", df_lime_repr),
                       ("Error maximo",   df_lime_err),
                       ("Discordante",    df_lime_disc)]:
        if df_g.empty:
            print(f"  {tag}: sin casos")
            continue
        n     = df_g["idx"].nunique()
        d_med = df_g.drop_duplicates("idx")["distancia"].mean()
        top_f = (df_g.groupby("feature")["peso_lime"]
                 .apply(lambda x: x.abs().mean()).idxmax())
        print(f"  {tag:<22}: {n} casos | dist.media={d_med:.2f} | feature clave: {top_f}")

else:
    print("LIME no disponible — instalar con: pip install lime")

## 8. TabNet — Análisis de atención

In [ ]:
# =============================================================================
# Si SHAP_PARA_TABNET = False, se usa el mecanismo de atención nativo.
# La atención de TabNet indica qué features usa el modelo en cada paso
# de decisión, siendo un sustituto interpretable de SHAP para esta arquitectura.
# =============================================================================

if not SHAP_PARA_TABNET:
    try:
        art_tn = cargar_pipeline("TabNet", SP_REFERENCIA)
        df_te_tn = cargar_split_parquet(SP_REFERENCIA, "test")
        feats_tn = art_tn["features"]
        X_tn = df_te_tn[[f for f in feats_tn if f in df_te_tn.columns]]
        X_tn = X_tn.reindex(columns=feats_tn)
        X_tn_imp = pd.DataFrame(art_tn["imp_num"].transform(X_tn), columns=feats_tn)
        X_tn_sc  = pd.DataFrame(art_tn["scaler"].transform(X_tn_imp), columns=feats_tn)

        # Obtener máscaras de atención
        clf_tn    = art_tn["modelo"]
        _, masks  = clf_tn.explain(X_tn_sc.values.astype(np.float32))
        # masks tiene shape: (n_pasos × n_muestras × n_features)
        # Importancia agregada: media sobre pasos y muestras
        imp_tn = np.abs(masks).mean(axis=(0, 1))
        imp_tn_series = pd.Series(imp_tn, index=feats_tn).sort_values(ascending=False)

        print("Top 10 features por atención TabNet:")
        for feat, val in imp_tn_series.head(10).items():
            et = ETIQUETAS_FEATURES.get(feat, feat)
            print(f"  {et:<35}: {val:.4f}  [{bloque_de(feat)}]")

        # Gráfico comparativo: SHAP (modelo principal) vs. Atención (TabNet)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        # Panel izquierdo: SHAP del modelo principal
        top_shap = importancias.head(15)
        etiq_shap = [ETIQUETAS_FEATURES.get(v, v) for v in top_shap.index]
        colores_s = [THEME.get("blocks",{}).get(bloque_de(v),"#888") for v in top_shap.index]
        axes[0].barh(etiq_shap[::-1], top_shap.values[::-1],
                     color=colores_s[::-1], edgecolor="white")
        axes[0].set_title(f"|SHAP| — {MODELO_XAI}", fontweight="bold")
        axes[0].set_xlabel("|SHAP| medio")

        # Panel derecho: Atención TabNet
        top_tn = imp_tn_series.head(15)
        etiq_tn = [ETIQUETAS_FEATURES.get(v, v) for v in top_tn.index]
        colores_t = [THEME.get("blocks",{}).get(bloque_de(v),"#888") for v in top_tn.index]
        axes[1].barh(etiq_tn[::-1], top_tn.values[::-1],
                     color=colores_t[::-1], edgecolor="white")
        axes[1].set_title("Atención — TabNet", fontweight="bold")
        axes[1].set_xlabel("Atención media")

        fig.suptitle(f"Importancia de features: {MODELO_XAI} vs. TabNet — {SP_REFERENCIA}",
                     fontweight="bold")
        save_figure(f"04_shap_vs_tabnet_{SP_REFERENCIA}")
        plt.show()

        # Guardar importancias TabNet
        imp_tn_series.reset_index().rename(
            columns={"index":"variable", 0:"atencion_tabnet"}
        ).to_csv(PATHS["FOLDER_RESULTS_TABLES"] /
                 f"tabnet_atencion_{SP_REFERENCIA}.csv", index=False)
        print("✓ Importancias TabNet guardadas")

    except FileNotFoundError:
        print("⚠ Pipeline TabNet no encontrado para el análisis de atención.")
    except Exception as e:
        print(f"⚠ Error en análisis de atención TabNet: {e}")
else:
    print("SHAP_PARA_TABNET = True: se incluye TabNet en el cálculo SHAP global.")

## 9. Análisis formal de errores de predicción

Caracterizacion cuantitativa de los casos donde el modelo falla con mayor
severidad. Esta seccion documenta los limites del predictor y responde
directamente a la exigencia metodologica del revisor.

In [ ]:
# =============================================================================
# Analisis formal de errores del modelo sobre el conjunto de prueba completo
# =============================================================================
print("=" * 60)
print(f"Errores formales — {MODELO_XAI} x {SP_REFERENCIA}")
print("=" * 60)

errores_ord_all = np.abs(y_pred_all - y_te)

print("\nDistribucion de la distancia ordinal |pred - real|:")
for dist in range(4):
    n   = (errores_ord_all == dist).sum()
    pct = n / len(y_te) * 100
    print(f"  Distancia {dist}: {n:>6,} ({pct:>5.1f}%)")

print("\nErrores graves (distancia >= 2):")
filas_err = []
for cp in range(4):
    for cr in range(4):
        dist = abs(cp - cr)
        if dist < 2:
            continue
        mask = (y_pred_all == cp) & (y_te == cr)
        n    = mask.sum()
        if n == 0:
            continue
        pct  = n / len(y_te) * 100
        tipo_error = "Sobreestimacion" if cp > cr else "Subestimacion"
        pais_top = ""
        if COL_PAIS in df_te.columns:
            top = df_te.loc[mask, COL_PAIS].value_counts().head(2)
            pais_top = ", ".join(f"{p}({c})" for p, c in top.items())
        filas_err.append({
            "clase_pred": cp, "clase_real": cr, "distancia": dist,
            "tipo": tipo_error, "n_casos": n, "pct_test": round(pct, 2),
            "paises_top": pais_top})
        print(f"  Pred={cp} Real={cr} ({tipo_error}): {n:,} ({pct:.2f}%) {pais_top}")

df_errores = pd.DataFrame(filas_err)
if not df_errores.empty:
    ruta_err = PATHS["FOLDER_RESULTS_TABLES"] / f"errores_graves_{MODELO_XAI}_{SP_REFERENCIA}.csv"
    df_errores.to_csv(ruta_err, index=False)
    print(f"\n✓ Tabla de errores guardada: {ruta_err.name}")

# Matriz de confusion con errores graves destacados
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm_arr = confusion_matrix(y_te, y_pred_all, normalize="true")
etiq_c = ["Para nada\nsat.", "No muy\nsat.", "Mas bien\nsat.", "Muy\nsat."]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_arr * 100, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=etiq_c, yticklabels=etiq_c, linewidths=0.4, ax=ax,
            cbar_kws={"label": "% por fila (real)"}, annot_kws={"size": 10})
for i in range(4):
    for j in range(4):
        if abs(i - j) >= 2:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                        edgecolor="#DC2626", lw=2.5))
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title(
    f"Matriz de confusion — {MODELO_XAI} x {SP_REFERENCIA}\n"
    "Contornos rojos: errores con distancia ordinal >= 2",
    fontweight="bold")
from utils.plots import save_figure
save_figure(f"04_confusion_errores_{MODELO_XAI}_{SP_REFERENCIA}")
plt.show()
print("✓ Matriz de confusion con errores graves guardada")

## 10. Guardado de valores SHAP y resumen

In [ ]:
# =============================================================================
# Verificar que todos los valores SHAP necesarios para NB05 están disponibles
# =============================================================================
print("Estado de valores SHAP disponibles para el análisis de estabilidad:")
for modelo in ["OLO","XGBoost","CatBoost","LightGBM"]:
    for sp in ["SP1","SP2","SP3"]:
        disp = shap_disponible(modelo, sp)
        print(f"  {modelo:<12} {sp}: {'✓' if disp else '✗ FALTA'}")

print()
print("Archivos generados:")
for f in sorted(PATHS["FOLDER_RESULTS_FIGURES"].glob("04_*.png")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*shap*")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_SHAP"].glob("*.parquet")):
    print(f"  results/shap/{f.name} ({f.stat().st_size//1024} KB)")